# VMD-MFGNN — Robustness Experiments, Colab Runner

This notebook closes 5 identified gaps in Paper 1 (VMD-MFGNN) before final
submission, all stress-testing the central "Adjacency Collapse" finding
(see `STATUS.md`, "Adjacency Collapse Diagnosed", 2026-08-09): does the
graph-embedding collapse depend on VMD's band count K, the decomposition
method, the training loss function, or the random seed -- or is it
invariant across all of them?

**What this notebook is NOT.** It does not modify, re-run, or risk the
existing, already-verified main pipeline (`results/`,
`notebooks/archive_paper1_vmd_mfgnn/vmd_mfgnn_v2_colab.ipynb`). All new
results go to `results/robustness/<experiment>/`, entirely separate
directories. All new code (`src/data_pipeline.py`'s `EMDDecomposer`/
`CEEMDANDecomposer`/`build_decomposed_modes`, `src/trainer.py`'s
`training.loss_fn` option, `src/hpo.py`'s widened search space,
`src/diagnostics.py`, `src/experiments.py`) is backward compatible: every
new option defaults to the existing, already-verified behavior when not
explicitly invoked (same pattern as `no_decay_graph_embeddings`/
`normalize_graph_embeddings` in `configs/default.yaml`).

**Sections:**
1. Setup (Drive mount, repo clone/pull, dependency install)
2. Data acquisition (reuse `data_pipeline.py`, shape/range sanity checks)
3. Experiment 1 -- VMD band-count (K) sweep, K in {3,5,7,9}
4. Experiment 2 -- Decomposition method comparison (VMD vs EMD, optional CEEMDAN)
5. Experiment 3 -- Loss function comparison (MSE vs MAE vs Huber)
6. Experiment 4 -- Multi-seed robustness (full_model vs pooled_graph_matched_dim, 5 seeds)
7. Experiment 5 -- Wider HPO pass (K + weight_decay + batch_size added, 30 trials)
8. Final aggregation: does the collapse finding depend on K / decomposition / loss / seed?
9. Package results for download

**Resumability design (read before running Section 3 onward).** Every
experiment cell (one K value, one decomposition method, one loss function,
one seed) is appended as ONE line to its own `results/robustness/<exp>/
*_results.jsonl` file IMMEDIATELY after it finishes (`src/experiments.py`'s
`run_resumable_grid`/`append_cell_result`). On restart, a cell counts as
done ONLY if its jsonl row exists AND its checkpoint file genuinely exists
on disk -- matching this project's own documented fix for the equivalent bug
in CuBench (`paper2/notebooks/cubench_colab.ipynb`; see git log "Fix resume
logic to require actual prediction files, not just log lines"). Drive sync
below **unconditionally overwrites** (no size/mtime "is this newer"
heuristic) -- deliberately avoiding the exact file-size-based staleness bug
class documented in `STATUS.md`/`docs/todo.txt` item 5 (Paper 1's OLD
notebook compared file sizes to decide whether to re-sync, which silently
kept a stale Drive copy for same-shaped `.npy` arrays). This mirrors
CuBench's fix for the identical problem, not a fresh design.

**Honesty note on wall-clock estimates.** Every section below states a
realistic estimate reasoned from this project's own logged timing (the main
notebook's one completed real run: `VMD-MFGNN training done in 1944.9s`
(~32 min) at `hidden_dim=128`; the graph-fix notebook's estimates built on
that same number; CuBench's logged per-family throughput). These are
planning numbers, not guarantees -- this notebook was built without GPU
access in the packaging environment, so none of Sections 3-7 have actually
been run end-to-end. Verification that WAS possible locally (CPU-only, no
GPU) is documented inline in Section 1 and in the task report.


## Section 1: Setup

**Realistic wall-clock estimate: 5-15 minutes** (network-bound: Drive mount,
git clone, `pip install`). Same order of magnitude as the main notebook's
Setup section.


### 1.1 Mount Google Drive (content-addressed sync, no size-based staleness check)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/Copper_Paper1'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive results root:', DRIVE_ROOT)


### 1.2 Get the repo

**IMPORTANT:** this notebook's new code (`src/experiments.py`,
`src/diagnostics.py`, the `EMDDecomposer`/loss-function/wider-HPO additions
to `src/data_pipeline.py`/`src/trainer.py`/`src/hpo.py`) must be **pushed to
the remote** (`https://github.com/anmol0705/Copper_Price_Forecasting`,
branch `graph-fix-experiment` as of this writing -- confirm with
`git branch --show-current` / `git log -1` locally before relying on a
fresh clone here) before this cell will see it. If you're running this right
after the code was written locally and haven't pushed yet, push first, or
use Option B (zip upload) instead.


In [ ]:
# ---- OPTION A: git clone/pull from the real remote (recommended) ----
REPO_URL = 'https://github.com/anmol0705/Copper_Price_Forecasting.git'
BRANCH = 'graph-fix-experiment'  # confirm this is still current: `git branch --show-current` locally

import subprocess
if not os.path.exists('/content/copper'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, '/content/copper'], check=True)
else:
    subprocess.run(['git', '-C', '/content/copper', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/copper', 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/copper', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
print('Repo ready at /content/copper on branch', BRANCH)


In [ ]:
# ---- OPTION B: upload a copper.zip (use if you haven't pushed yet) ----
# 1. Zip the repo locally: `cd D:\copper && zip -r copper.zip . -x '.git/*'`
# 2. Run this cell, click "Choose Files", select copper.zip.
# 3. Uncomment the extraction lines.

# from google.colab import files
# uploaded = files.upload()  # select copper.zip
# import zipfile
# with zipfile.ZipFile('copper.zip', 'r') as zf:
#     zf.extractall('/content/copper')
# print('Extracted to /content/copper')


In [ ]:
import os, sys
assert os.path.isdir('/content/copper'), (
    "Repo not found at /content/copper -- run Option A or Option B above first."
)
os.chdir('/content/copper/paper1')  # all paths below (data/, results/, configs/) are relative to paper1/
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())
print(sorted(os.listdir('.')))


### 1.3 Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q vmdpy optuna yfinance torch-geometric EMD-signal


### 1.4 Confirm runtime, sanity-import the new modules

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected -- Colab Runtime > Change runtime type > GPU. '
          'CPU-only will still run correctly, just much slower for the K-sweep/'
          'decomposition/loss/multiseed experiments (item 5, wider HPO, is the '
          'most CPU-sensitive since it runs many short trials).')

from src.utils import load_config
from src.data_pipeline import create_datasets, EMDDecomposer, CEEMDANDecomposer
from src.trainer import VMDMFGNNTrainer
from src.diagnostics import run_full_diagnostic
from src import experiments as exp
from src import hpo as hpo_mod
print('All new modules import cleanly.')


## Section 2: Data Acquisition

Reuses `src/data_pipeline.py` unchanged for the base VMD (K=5) dataset --
this is the SAME data build the main pipeline already used and verified,
not a new data path. Per-experiment K/decomposition-method variants each
get their OWN cache file (`data/vmd_modes_K{K}.npy`,
`data/{method}_modes_K{K}.npy`) and are built lazily inside Sections 3-4,
not here.

**Realistic wall-clock estimate: 1-3 min (warm cache, e.g. restored from
Drive) or 15-45 min (cold, per the main notebook's documented VMD
decomposition timing).**


In [ ]:
config = load_config('configs/default.yaml')
print('Base config vmd.K:', config['vmd']['K'])
print('Base config training.loss_fn:', config['training'].get('loss_fn', 'mse (default)'))

base_data = create_datasets(config)
print(f"Variables: {base_data['variable_names']}")
print(f"Train/Val/Test samples: {len(base_data['train_ds'])}/{len(base_data['val_ds'])}/{len(base_data['test_ds'])}")
print(f"num_vars={base_data['num_vars']}, num_modes={base_data['num_modes']}")

# Shape/range sanity check against the known-good original run (see STATUS.md /
# results/archive_paper1/all_results.json for the reference scale): 8 variables,
# K=5 modes, ~4000 trading days total.
assert base_data['num_vars'] == 8, f"expected 8 variables, got {base_data['num_vars']}"
assert base_data['num_modes'] == config['vmd']['K']
x0, y0 = base_data['train_ds'][0]
print('Sample X shape:', tuple(x0.shape), 'Y shape:', tuple(y0.shape))
assert x0.shape == (config['data']['lookback'], config['vmd']['K'], base_data['num_vars'])
print('Shape checks passed.')


## Shared Drive-sync + progress-checker helpers

Same pattern as `paper2/notebooks/cubench_colab.ipynb`: unconditional
overwrite (no size/mtime staleness heuristic -- see Section-1 note above),
synced per-experiment jsonl AND the raw checkpoint/prediction artifacts
(not just the jsonl summary), so a Colab disconnect never loses more than
the in-flight cell.


In [ ]:
import shutil, time as _time

def sync_to_drive(local_dir, note=''):
    """Unconditionally copies `local_dir` (a results/robustness/<exp> dir,
    containing both the *_results.jsonl and its checkpoints/ subdir) to the
    matching path under DRIVE_ROOT. Always overwrites -- deliberately no
    size/mtime freshness check (see Section 1's staleness-bug note)."""
    if not os.path.isdir(local_dir):
        print(f'{local_dir} does not exist yet, nothing to sync')
        return
    rel = os.path.relpath(local_dir, 'results')
    dest = os.path.join(DRIVE_ROOT, rel)
    os.makedirs(os.path.dirname(dest) or '.', exist_ok=True)
    shutil.copytree(local_dir, dest, dirs_exist_ok=True)
    print(f'[{_time.strftime("%H:%M:%S")}] synced {local_dir} -> {dest} {note}')


def restore_from_drive_if_present(local_dir):
    """Opposite direction: pulls a prior Drive-backed copy down into the
    fresh clone's local_dir, if one exists, so a resumed session picks up
    completed cells from before a disconnect. Always overwrites local with
    the Drive copy -- same no-staleness-heuristic principle."""
    rel = os.path.relpath(local_dir, 'results')
    src = os.path.join(DRIVE_ROOT, rel)
    if os.path.isdir(src):
        os.makedirs(local_dir, exist_ok=True)
        shutil.copytree(src, local_dir, dirs_exist_ok=True)
        print(f'Restored {src} -> {local_dir}')
    else:
        print(f'No prior Drive backup at {src}, starting fresh')


def jsonl_progress(jsonl_path):
    import json as _json
    if not os.path.exists(jsonl_path):
        print(f'{jsonl_path}: no results yet.')
        return {}
    seen, errors = {}, 0
    with open(jsonl_path) as f:
        for line in f:
            try:
                r = _json.loads(line)
            except Exception:
                continue
            key = tuple(r.get('cell_key', []))
            seen[key] = r
            if 'error' in r:
                errors += 1
    print(f'{jsonl_path}: {len(seen)} completed cells (errored: {errors})')
    for k in sorted(seen):
        print('  ', k)
    return seen


def load_rows_from_jsonl(jsonl_path):
    """Reads ALL rows from a *_results.jsonl file directly off disk (via
    exp.load_completed_cells, with the SAME completed-checkpoint artifact
    check every experiment uses), independent of whatever is or isn't still
    in this kernel's memory. Section 8 uses this (not the in-memory
    k_sweep_results/decomposition_results/etc. variables) specifically so
    the final aggregation works even if you're resuming a fresh kernel
    session and only want to view already-completed results, without
    re-running every Section 3-7 cell first."""
    completed = exp.load_completed_cells(
        jsonl_path, artifact_check_fn=lambda row: exp._checkpoint_is_complete(
            row.get('checkpoint_path', '')))
    return list(completed.values())


## Section 3: Experiment 1 — VMD Band-Count (K) Sweep

Retrains full VMD-MFGNN at K in {3, 5, 7, 9}, using the already-tuned other
hyperparameters (`results/archive_paper1/hpo_best_params.json`), and runs
the SAME gradient/adjacency diagnostic used to find the original bug
(`src/diagnostics.py::run_full_diagnostic`, reusing STATUS.md's exact
methodology: RMS embedding magnitude vs. Xavier-init reference, softmax
spread, fraction of kept edges within tolerance of 1/N, real
forward/backward gradient norms on emb1/emb2, and embedding-direction
cosine similarity vs. a fresh init). Directly tests whether the collapse is
K-dependent or invariant.

**Resumable:** each K is one jsonl cell (own checkpoint + own VMD-modes
cache), so a disconnect resumes at the next incomplete K.

**Realistic wall-clock estimate:** ~30-40 min one-time CPU cost per NEW K's
VMD decomposition (per `VMDDecomposer`'s docstring: "~30-40 minutes one-time
CPU for the full ~4,000-day x 10-variable dataset" -- unaffected by the
2026-08-07 8-variable scope reduction) + ~20-40 min training per K (per the
main notebook's ~32 min real observed run) + <1 min diagnostic. **4 K
values x (~35 min decomposition + ~30 min training) = roughly 4-4.5 hours**
if all 4 K-value caches are cold; **~2 hours** if K=5's cache can be reused
from the main pipeline's `data/vmd_modes.npy` (only 3 NEW decompositions
needed, K=3/7/9). Budget for at least one disconnect; resume picks up at
the next incomplete K.


In [ ]:
K_SWEEP_DIR = 'results/robustness/k_sweep'
restore_from_drive_if_present(K_SWEEP_DIR)
jsonl_progress(f'{K_SWEEP_DIR}/k_sweep_results.jsonl')


In [ ]:
k_sweep_results = exp.run_k_sweep(
    config, k_values=[3, 5, 7, 9],
    hpo_best_params_path='results/archive_paper1/hpo_best_params.json',
    results_dir=K_SWEEP_DIR,
)
sync_to_drive(K_SWEEP_DIR, note='after K-sweep')


In [ ]:
# Quick per-K summary: h1 RMSE/MAE/DA + collapse verdict + emb gradient norm.
print(f"{'K':>3} | {'h1_rmse':>9} | {'h1_da':>7} | {'collapse_verdict':<45} | {'emb_grad_norm':>14}")
print('-' * 95)
for row in k_sweep_results:
    k = row['K']
    tm = row['test_metrics']
    diag = row['diagnostic']
    h1 = tm.get('h1', {})
    verdict = diag['adjacency_state']['verdict']
    grad = diag['gradient_magnitude']['emb_grad_norm_mean']
    print(f"{k:>3} | {h1.get('rmse', float('nan')):>9.5f} | {h1.get('da', float('nan')):>6.2f}% | "
          f"{verdict:<45} | {grad:>14.6e}")


## Section 4: Experiment 2 — Alternative Decomposition Method Comparison

Retrains full model with VMD replaced by `EMDDecomposer` (item 2), at the
SAME K as the main config (see `EMDDecomposer`'s docstring for exactly how
K interacts with EMD's data-driven natural IMF count: truncate-and-sum the
tail if EMD naturally produces more IMFs than K, zero-pad if fewer -- this
is disclosed, not hidden). Also runs the same gradient/adjacency
diagnostic.

**CEEMDAN (optional, NOT run by default):** implemented
(`CEEMDANDecomposer`), and separately leakage-verified locally at small
scale (`tests/test_emd_leakage.py`'s `CEEMDANDecomposer` check: T=120,
rolling_window=40, trials=5 -- reduced scale for speed, see the timing
notes below). Two REAL bugs were found and fixed during that local
verification, not just theorized:
  1. PyEMD's CEEMDAN defaults to `parallel=True`, spawning a fresh
     `multiprocessing` worker pool PER WINDOW -- on Windows (spawn-based
     process creation), this overhead is so severe that an initial test at
     this same tiny scale did not finish within 6 minutes before being
     killed. Fixed by defaulting `CEEMDANDecomposer(parallel=False)`.
  2. PyEMD's CEEMDAN does not seed its noise RNG by default, so
     decomposing the identical window twice gives DIFFERENT results --
     the first (post-fix-1) run of the leakage test genuinely FAILED
     because of this (not a real leak: the "prefix differs" signal was
     pure noise-seed nondeterminism). Fixed by seeding
     `noise_seed(random_seed_base + t)` per window, making results
     reproducible.
After both fixes, the local leakage test passes cleanly and quickly. Still
deliberately excluded from the default `methods` list below due to a
disclosed compute cost constraint -- true daily-refit CEEMDAN at
`trials=20` is still roughly a 20x per-window cost multiplier over plain
EMD even with the multiprocessing fix, which would push one-time CPU cost
from tens of minutes into many hours for the full dataset (see
`CEEMDANDecomposer`'s docstring). Uncomment `'ceemdan'` in the `methods`
list below ONLY if you have a much larger time budget than a single Colab
session.

**Realistic wall-clock estimate:** EMD decomposition is generally faster
per-window than VMD's ADMM optimization, but this is still a true
daily-refit O(T) x O(sift) computation over ~4,000 days x 8 variables --
budget the same ~30-40 min order of magnitude as VMD's documented cost,
plus ~20-40 min training + diagnostic. **~1-1.5 hours for VMD+EMD** (VMD
reuses the main pipeline's existing cache if present, so realistically
closer to the EMD-only cost, ~45min-1.25h). Adding CEEMDAN would add many
additional hours (see cost note above) -- not included in this estimate.


In [ ]:
DECOMP_DIR = 'results/robustness/decomposition'
restore_from_drive_if_present(DECOMP_DIR)
jsonl_progress(f'{DECOMP_DIR}/decomposition_results.jsonl')


In [ ]:
# methods = ['vmd', 'emd', 'ceemdan']  # uncomment to also run CEEMDAN (see cost note above)
methods = ['vmd', 'emd']

decomposition_results = exp.run_decomposition_comparison(
    config, methods=methods,
    hpo_best_params_path='results/archive_paper1/hpo_best_params.json',
    results_dir=DECOMP_DIR,
)
sync_to_drive(DECOMP_DIR, note='after decomposition comparison')


In [ ]:
print(f"{'method':>8} | {'eff_K':>6} | {'h1_rmse':>9} | {'h1_da':>7} | {'collapse_verdict':<45}")
print('-' * 90)
for row in decomposition_results:
    tm = row['test_metrics']
    diag = row['diagnostic']
    h1 = tm.get('h1', {})
    print(f"{row['method']:>8} | {row['effective_K']:>6} | {h1.get('rmse', float('nan')):>9.5f} | "
          f"{h1.get('da', float('nan')):>6.2f}% | {diag['adjacency_state']['verdict']:<45}")


## Section 5: Experiment 3 — Loss Function Comparison

Retrains full model under `training.loss_fn` in {"mse", "mae", "huber"}
(item 3; `src/trainer.py`'s `_LOSS_FNS` dispatch table, locally smoke-tested
-- see task report for the actual CPU-only test output confirming the
right `F.*` is invoked and gradients differ between choices). Since the
data build doesn't depend on the loss function, this reuses ONE data build
across all three loss choices -- only the trainer's loss changes.

**Realistic wall-clock estimate:** no new decomposition cost (reuses the
main K=5 VMD cache) -- just 3 training runs at ~20-40 min each + diagnostic.
**~1-2 hours total.**


In [ ]:
LOSS_DIR = 'results/robustness/loss_comparison'
restore_from_drive_if_present(LOSS_DIR)
jsonl_progress(f'{LOSS_DIR}/loss_comparison_results.jsonl')


In [ ]:
loss_results = exp.run_loss_comparison(
    config, loss_fns=['mse', 'mae', 'huber'],
    hpo_best_params_path='results/archive_paper1/hpo_best_params.json',
    results_dir=LOSS_DIR,
)
sync_to_drive(LOSS_DIR, note='after loss comparison')


In [ ]:
print(f"{'loss_fn':>8} | {'h1_rmse':>9} | {'h1_da':>7} | {'collapse_verdict':<45}")
print('-' * 80)
for row in loss_results:
    tm = row['test_metrics']
    diag = row['diagnostic']
    h1 = tm.get('h1', {})
    print(f"{row['loss_fn']:>8} | {h1.get('rmse', float('nan')):>9.5f} | "
          f"{h1.get('da', float('nan')):>6.2f}% | {diag['adjacency_state']['verdict']:<45}")


## Section 6: Experiment 4 — Multi-Seed Robustness Check

Per STATUS.md's already-recommended "middle path": re-runs ONLY
`full_model` and `pooled_graph_matched_dim` (the two ablation rows carrying
the paper's central RQ1 claim) across 5 seeds each (42-46), reporting mean
+/- std RMSE at all 4 horizons, plus a paired significance test
(`src/utils.py::paired_seed_significance_test` -- paired t-test + Wilcoxon
signed-rank cross-check across the 5 seeds) for whether the full-model-vs-
pooled-graph ranking is stable or seed-dependent noise.

**Realistic wall-clock estimate:** no new decomposition cost. 2 variants x
5 seeds x ~20-40 min/run = **~3.5-7 hours**, the single largest time cost in
this notebook (matches STATUS.md's own estimate: "~1.2-3.7 extra Colab
hours" for this exact middle-path design -- our estimate is somewhat higher
since it also runs the full diagnostic per seed, not just metrics). Budget
for at least one, likely multiple, disconnects; each (variant, seed) cell
resumes independently.

**DISCLOSURE:** both variants here train at the TUNED hyperparameters
(`results/archive_paper1/hpo_best_params.json`), NOT the untuned
`hidden_dim=64` config `results/ablation_results.json`'s existing rows use.
This section's numbers are therefore NOT directly comparable to the main
ablation table -- see `run_multiseed`'s docstring in `src/experiments.py`
for the full reasoning. State this explicitly in any paper text drawing on
this section, rather than silently cross-referencing the two tables.


In [ ]:
MULTISEED_DIR = 'results/robustness/multiseed'
restore_from_drive_if_present(MULTISEED_DIR)
jsonl_progress(f'{MULTISEED_DIR}/multiseed_results.jsonl')


In [ ]:
multiseed_output = exp.run_multiseed(
    config, seeds=[42, 43, 44, 45, 46],
    variants=['full_model', 'pooled_graph_matched_dim'],
    hpo_best_params_path='results/archive_paper1/hpo_best_params.json',
    results_dir=MULTISEED_DIR,
)
sync_to_drive(MULTISEED_DIR, note='after multiseed')


In [ ]:
print('=== Mean +/- std RMSE per variant/horizon ===')
for variant, per_h in multiseed_output['aggregate'].items():
    for h, stats_ in per_h.items():
        print(f"  {variant:28s} {h}: {stats_['rmse_mean']:.5f} +/- {stats_['rmse_std']:.5f} "
              f"(n={stats_['n_seeds']})")

print('\n=== Paired significance test (full_model vs pooled_graph_matched_dim), per horizon ===')
for h, sig in multiseed_output['significance'].items():
    print(f"  {h}: mean_diff={sig['mean_diff']:.5f}  "
          f"t_pvalue={sig['t_pvalue']:.4f}  wilcoxon_pvalue={sig['wilcoxon_pvalue']:.4f}")
print('\n(n=5 seeds is a very small sample -- treat p-values as indicative, not definitive; '
      'see the paired_seed_significance_test docstring for the honesty caveat on both tests'
      ' at this sample size.)')


## Section 7: Experiment 5 — Wider HPO Pass

Extends the Optuna search space (`src/hpo.py::run_hpo`) to also include K
(categorical {3,5,7,9}), `weight_decay` (log-uniform 1e-6 to 1e-3), and
`batch_size` (categorical {16,32,64}), on top of the original 5 parameters
(hidden_dim/num_heads/learning_rate/dropout/num_gnn_layers). `n_trials=30`
(vs. the original 10) -- chosen as a realistic Colab time-budget compromise:
enough trials for TPE to meaningfully explore an 8-dimensional space without
pushing this already-long notebook's single largest additional item past a
half-day. This does NOT replace or overwrite the original tuned
hyperparameters used by Sections 3-6 above (written to
`results/robustness/wider_hpo/hpo_wider_best_params.json`, never
`results/hpo_best_params.json`) -- it's a separate, complementary check for
whether a genuinely wider search finds a meaningfully better configuration.

**Realistic wall-clock estimate:** each trial trains for `trial_epochs=25`
(reduced budget, same as the original HPO's convention) at whichever
sampled hidden_dim/K/batch_size -- roughly 2-6 min/trial depending on
hidden_dim and whether MedianPruner cuts it short. Pre-building the 4 K
datasets (`build_data_for_k_values`) costs the same ~30-40 min/K
decomposition as Section 3 for any K not already cached from Section 3 (if
Section 3 already ran, all 4 K caches are warm and this cost is ~0).
**30 trials x ~2-6 min = ~1-3 hours**, PLUS up to ~2 hours of K-decomposition
cache-building if Section 3 has not already run (run Section 3 first to
avoid double-paying this cost).


In [ ]:
WIDER_HPO_DIR = 'results/robustness/wider_hpo'
restore_from_drive_if_present(WIDER_HPO_DIR)


In [ ]:
# Reuses K-sweep's cached per-K datasets if Section 3 already ran (same
# cache paths, data/vmd_modes_K{K}.npy) -- run Section 3 first to avoid
# double-paying the K-decomposition cost here.
wider_best_params = exp.run_wider_hpo(
    config, k_values=[3, 5, 7, 9], n_trials=30, trial_epochs=25,
    results_dir=WIDER_HPO_DIR,
)
sync_to_drive(WIDER_HPO_DIR, note='after wider HPO')
print('Wider-HPO winning hyperparameters:', wider_best_params)


In [ ]:
import pandas as pd
trials_df = pd.read_csv(f'{WIDER_HPO_DIR}/hpo_wider_trials.csv')
print(f'{len(trials_df)} trials completed.')
print(trials_df[[c for c in trials_df.columns if c.startswith('params_') or c == 'value']].sort_values('value').head(10))


## Section 8: Final Aggregation — Does the Collapse Finding Depend on K / Decomposition / Loss / Seed?

Pulls together every diagnostic verdict from Sections 3-6 into one summary
table, directly answering the question this whole notebook exists to ask.
A finding is "INVARIANT" (collapse present regardless of the axis tested) if
every cell's `adjacency_state` verdict is COLLAPSED; "AXIS-DEPENDENT" if
verdicts differ across the axis; "RESOLVED" if every cell shows NOT
COLLAPSED. Section 7 (wider HPO) is reported separately since it answers a
different question (does a wider search find better hyperparameters at
all, not specifically about the collapse mechanism).

**This section reads every experiment's result directly from its
`*_results.jsonl` file on disk** (`load_rows_from_jsonl`, defined in the
shared-helpers cell above), NOT from the `k_sweep_results`/
`decomposition_results`/`loss_results`/`multiseed_output` in-memory
variables Sections 3-6 assigned. This means Section 8 works correctly even
if you're resuming a fresh kernel session days later and only want the
final summary, without re-running every Section 3-6 cell first (as long as
the underlying jsonl + checkpoint files are present locally or restored
from Drive).


In [ ]:
K_SWEEP_JSONL = f'{K_SWEEP_DIR}/k_sweep_results.jsonl'
DECOMP_JSONL = f'{DECOMP_DIR}/decomposition_results.jsonl'
LOSS_JSONL = f'{LOSS_DIR}/loss_comparison_results.jsonl'
MULTISEED_JSONL = f'{MULTISEED_DIR}/multiseed_results.jsonl'

def collapse_summary_row(axis_name, cell_label, diag):
    state = diag['adjacency_state']
    grad = diag['gradient_magnitude']
    move = diag['adjacency_movement']
    return {
        'axis': axis_name, 'value': cell_label,
        'verdict': state['verdict'],
        'n_ok': state['n_ok'], 'n_total': state['n_total'],
        'emb_grad_norm_mean': grad['emb_grad_norm_mean'],
        'other_grad_norm_mean': grad['other_grad_norm_mean'],
        'mean_cosine_similarity_vs_init': move['mean_cosine_similarity'],
    }

summary_rows = []
for row in load_rows_from_jsonl(K_SWEEP_JSONL):
    summary_rows.append(collapse_summary_row('K', row['K'], row['diagnostic']))
for row in load_rows_from_jsonl(DECOMP_JSONL):
    summary_rows.append(collapse_summary_row('decomposition_method', row['method'], row['diagnostic']))
for row in load_rows_from_jsonl(LOSS_JSONL):
    summary_rows.append(collapse_summary_row('loss_fn', row['loss_fn'], row['diagnostic']))
for row in load_rows_from_jsonl(MULTISEED_JSONL):
    label = f"{row['variant']}_seed{row['seed']}"
    summary_rows.append(collapse_summary_row('seed_x_variant', label, row['diagnostic']))

import pandas as pd
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

collapsed_count = (summary_df['n_ok'] == 0).sum()
not_collapsed_count = (summary_df['n_ok'] == summary_df['n_total']).sum()
total = len(summary_df)
print(f'\n{collapsed_count}/{total} cells fully COLLAPSED, {not_collapsed_count}/{total} fully NOT COLLAPSED.')
if collapsed_count == total:
    print('=== HEADLINE: collapse finding is INVARIANT across K / decomposition / loss / seed. ===')
elif not_collapsed_count == total:
    print('=== HEADLINE: no cell shows the collapse pattern -- inconsistent with the original '
          'diagnosis; inspect summary_df manually before trusting this. ===')
else:
    print('=== HEADLINE: MIXED -- collapse is AXIS-DEPENDENT for at least one tested axis. '
          'Inspect summary_df to see which axis/value differs. ===')

summary_df.to_csv('results/robustness/collapse_invariance_summary.csv', index=False)
sync_to_drive('results/robustness', note='after final aggregation')


### Figure: collapse verdict / gradient-norm-to-emb across every axis

A single figure so the invariance-or-not finding is visually obvious, not
just a table.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)
axis_order = ['K', 'decomposition_method', 'loss_fn', 'seed_x_variant']
for ax, axis_name in zip(axes, axis_order):
    sub = summary_df[summary_df['axis'] == axis_name]
    if len(sub) == 0:
        ax.set_title(f'{axis_name} (no data)')
        continue
    ax.bar(sub['value'].astype(str), sub['emb_grad_norm_mean'])
    ax.set_title(axis_name)
    ax.set_xlabel(axis_name)
    ax.tick_params(axis='x', rotation=45)
axes[0].set_ylabel('mean emb1/emb2 gradient norm')
fig.suptitle('emb1/emb2 gradient magnitude across every tested axis')
fig.tight_layout()
os.makedirs('results/robustness/figures', exist_ok=True)
fig.savefig('results/robustness/figures/collapse_invariance_grad_norms.png', dpi=150)
plt.show()
sync_to_drive('results/robustness/figures', note='after figure')


## Section 9: Package Results for Download

Zips everything needed to reproduce every number and figure above without
retraining (jsonl grids, checkpoints, the aggregation CSV, the figure), then
copies the zip to Drive as well so it survives even if not downloaded
immediately -- same final-packaging pattern as CuBench's notebook.


In [ ]:
import shutil

shutil.make_archive('/content/robustness_results', 'zip', 'results/robustness')
print('Packaged: /content/robustness_results.zip')
print('Size (MB):', os.path.getsize('/content/robustness_results.zip') / 1e6)

shutil.copy2('/content/robustness_results.zip', os.path.join(DRIVE_ROOT, 'robustness_results.zip'))
print('Also copied to Drive:', os.path.join(DRIVE_ROOT, 'robustness_results.zip'))

from google.colab import files
files.download('/content/robustness_results.zip')


---
### Done

Check, in order:
1. Section 8's headline verdict (INVARIANT / MIXED / RESOLVED) -- this is
   the answer to the question this notebook exists to ask.
2. `results/robustness/collapse_invariance_summary.csv` and
   `results/robustness/figures/collapse_invariance_grad_norms.png` for the
   full per-cell detail.
3. `results/robustness/wider_hpo/hpo_wider_best_params.json` +
   `hpo_wider_trials.csv` for whether a wider search found a meaningfully
   different/better configuration than the original 10-trial HPO.
4. `results/robustness/multiseed/multiseed_results.jsonl` (raw per-seed
   metrics) if you want to re-derive the significance test independently of
   this notebook's own computation.
5. None of this touches or needs to be reconciled with the main pipeline's
   `results/` -- entirely separate, as documented in the overview cell.
